# Benchmark Comparison

This notebook compares all SISR model families across the two key axes:
1. **PSNR/SSIM** (quantitative fidelity)
2. **Perceptual quality** (LPIPS)

It also demonstrates using the `inference.benchmark` module to run evaluations.

In [ ]:
import sys
sys.path.insert(0, '..')

import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

import models
from metrics import compute_psnr, compute_ssim
from inference.tiled_inference import tiled_infer

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

## Synthetic Benchmark

We generate synthetic test images and compare all models on PSNR/SSIM.
(For real benchmarks, point to your Set5/Set14/BSD100/Urban100 data.)

In [ ]:
torch.manual_seed(0)
SCALE = 4
N_TEST = 8
HR_SIZE = 128
LR_SIZE = HR_SIZE // SCALE

hr_test = torch.rand(N_TEST, 3, HR_SIZE, HR_SIZE)
lr_test = F.interpolate(hr_test, scale_factor=1/SCALE, mode='bicubic', align_corners=False).clamp(0, 1)

print(f'Test set: {N_TEST} images | LR: {LR_SIZE}² → HR: {HR_SIZE}²')

In [ ]:
from models.interpolation.classical import BicubicSR
from models.cnn.srcnn import SRCNN
from models.cnn.edsr import EDSR
from models.cnn.rcan import RCAN
from models.cnn.espcn import ESPCN
from models.lightweight.imdn import IMDN
from models.transformer.swinir import SwinIR
from models.transformer.hat import HAT

# Small configs for demo — increase sizes for real training
MODELS = {
    'Bicubic':   BicubicSR(scale=SCALE),
    'SRCNN':     SRCNN(scale=SCALE, in_channels=3, out_channels=3),
    'EDSR':      EDSR(scale=SCALE, num_features=32, num_resblocks=4),
    'RCAN':      RCAN(scale=SCALE, num_features=32, num_groups=3, num_rcab_per_group=4),
    'ESPCN':     ESPCN(scale=SCALE, in_channels=3, out_channels=3, num_features=32),
    'IMDN':      IMDN(scale=SCALE, num_features=32, num_blocks=3),
    'SwinIR':    SwinIR(scale=SCALE, embed_dim=32, depths=[2,2], num_heads=[2,2]),
    'HAT':       HAT(scale=SCALE, embed_dim=48, depths=[2,2], num_heads=[2,2], window_size=8),
}

for name, m in MODELS.items():
    MODELS[name] = m.eval().to(device)

In [ ]:
results = {}

with torch.no_grad():
    for name, model in MODELS.items():
        psnrs, ssims = [], []
        for i in range(N_TEST):
            lr = lr_test[i:i+1].to(device)
            hr = hr_test[i:i+1]
            sr = model(lr).cpu()
            psnrs.append(compute_psnr(sr, hr, y_channel_only=False))
            ssims.append(compute_ssim(sr, hr, y_channel_only=False))
        results[name] = {
            'psnr': np.mean(psnrs),
            'ssim': np.mean(ssims),
            'params': sum(p.numel() for p in model.parameters() if p.requires_grad),
        }
        print(f'{name:<12} PSNR: {results[name]["psnr"]:6.2f} dB  SSIM: {results[name]["ssim"]:.4f}  Params: {results[name]["params"]:,}')

## PSNR vs. Parameter Count

In [ ]:
FAMILY_COLORS = {
    'Bicubic':  'gray',
    'SRCNN':    'blue',
    'EDSR':     'royalblue',
    'RCAN':     'darkblue',
    'ESPCN':    'green',
    'IMDN':     'limegreen',
    'SwinIR':   'orange',
    'HAT':      'red',
}

fig, ax = plt.subplots(figsize=(10, 6))

for name, m in results.items():
    ax.scatter(m['params'] / 1e6, m['psnr'], s=100,
               color=FAMILY_COLORS.get(name, 'black'), zorder=5)
    ax.annotate(name, (m['params'] / 1e6, m['psnr']),
                textcoords='offset points', xytext=(5, 5), fontsize=9)

ax.set_xlabel('Parameters (M)')
ax.set_ylabel('PSNR (dB)')
ax.set_title('PSNR vs. Model Size (×4 SR on synthetic test set)')
ax.grid(True, alpha=0.3)

legend_patches = [
    mpatches.Patch(color='gray', label='Interpolation'),
    mpatches.Patch(color='blue', label='CNN (PSNR)'),
    mpatches.Patch(color='green', label='Lightweight'),
    mpatches.Patch(color='orange', label='Transformer'),
]
ax.legend(handles=legend_patches)
plt.tight_layout()
plt.show()

## Visual Quality Comparison

In [ ]:
SHOW_MODELS = ['Bicubic', 'EDSR', 'SwinIR', 'HAT']
idx = 0

lr_img = lr_test[idx:idx+1].to(device)
hr_img = hr_test[idx]

sr_images = {}
with torch.no_grad():
    for name in SHOW_MODELS:
        sr_images[name] = MODELS[name](lr_img).cpu().squeeze(0)

n = len(SHOW_MODELS) + 2  # LR + models + HR
fig, axes = plt.subplots(1, n, figsize=(4*n, 4))

def show_img(ax, t, title):
    if t.ndim == 4:
        t = t.squeeze(0)
    ax.imshow(t.permute(1, 2, 0).clamp(0, 1).numpy())
    ax.set_title(title, fontsize=9)
    ax.axis('off')

show_img(axes[0], F.interpolate(lr_img.cpu(), scale_factor=SCALE, mode='nearest'), 'LR (nearest)')
for i, name in enumerate(SHOW_MODELS):
    p = compute_psnr(sr_images[name].unsqueeze(0), hr_img.unsqueeze(0), y_channel_only=False)
    show_img(axes[i+1], sr_images[name], f'{name}\nPSNR: {p:.2f} dB')
show_img(axes[-1], hr_img, 'HR (Ground Truth)')

plt.tight_layout()
plt.show()

## Running Real Benchmarks

After training your models, evaluate on standard benchmarks:

```bash
# Evaluate on Set5, Set14, BSD100, Urban100
python test.py \
    --config configs/edsr_x4.yaml \
    --checkpoint outputs/edsr_x4/best.pth \
    --datasets Set5 Set14 BSD100 Urban100

# Or use the benchmark runner to compare multiple models:
python -m inference.benchmark \
    --config configs/edsr_x4.yaml \
    --checkpoint outputs/edsr_x4/best.pth \
    --datasets Set5 Set14
```

## Published SOTA Results (×4)

| Model | Set5 PSNR | Set14 PSNR | Urban100 PSNR |
|-------|-----------|------------|---------------|
| Bicubic | 28.42 | 26.00 | 23.14 |
| SRCNN | 30.48 | 27.50 | 24.52 |
| EDSR | 32.46 | 28.58 | 26.64 |
| RCAN | 32.63 | 28.87 | 26.82 |
| SwinIR | 32.72 | 28.94 | 27.45 |
| HAT | 32.92 | 29.05 | 27.97 |

*Results from original papers. All models trained on DIV2K.*